In [3]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter

Welcome to JupyROOT 6.26/04


In [4]:
from math import sqrt

# Function to calculate the central branching ratio value
def calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    return Br
    
def calculate_br_ratio(Nsig,Nsig_err,  Nref,Nref_err, eff_sig, eff_ref):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br_ratio = (N_real_sig / N_real_ref)
    Br_ratio_err = (eff_ref/eff_sig) * math.sqrt( (Nsig_err / Nref)**2 + (Nsig / (Nref**2) * Nref_err)**2 )
    print(f'Br_ratio value = {Br_ratio:.4e}')
    print(f'Br_ratio Statistical uncertainty = {Br_ratio_err:.4e}')
    return Br_ratio, Br_ratio_err

# Function to calculate statistical uncertainty
def calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value):
    variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    return sqrt(variance) * central_value

# Function to print results in a formatted way
def print_results(label, central_value, stat_unc, Br_sig_dec):
    pull = (central_value - Br_sig_dec) / stat_unc
    print(f'{label} Central value = {central_value:.4e}')
    print(f'{label} Statistical uncertainty = {stat_unc:.4e}')
    print(f'{label} Pull = {pull:.4f}')
    print(f'{label} stas. unc./Central value = {stat_unc/central_value:.4e}\n')

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

def calculate_sig_eff_err(eff, N_gen):

    error = math.sqrt(eff * (1 - eff) / N_gen)
    return error

In [5]:
def cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref):
    eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)
    eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
    print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")
    
    eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal
    
    central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec)
    stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
    print_results(f"Mode: eta -> {mode}", central_value_1, stat_unc_1, Br_sig_dec)
    
    ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)
    
    return ratio_central_Br_gg, ratio_err_Br_gg

## Br(D+ -> eta K+)

### eta -> gg

In [6]:
# Constants
Br_ref_dec = 0.003770000
Br_sig_dec = 0.000125000

# Br_sig_PDG = 0.0001250000
# Br_sig_PDG_err = 0.0000160000

mode="gg"
# 0.91
eff_sig_cal =  0.045523
eff_ref_cal =  0.063126

Nsig_err, Nsig, Nref_err, Nref = 43.23205936648267, 533.8335242441342 , 210.14669520339157 , 22406.767936898214

print(f"Original values")
orig_val, orig_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

Original values
signal eff error: 8.509863e-05, ref eff error: 9.928168e-05
Mode: eta -> gg Central value = 1.2455e-04
Mode: eta -> gg Statistical uncertainty = 1.0154e-05
Mode: eta -> gg Pull = -0.0443
Mode: eta -> gg stas. unc./Central value = 8.1525e-02

Br_ratio value = 3.3037e-02
Br_ratio Statistical uncertainty = 2.6934e-03


In [7]:
(1.0155e-05-1.0269e-05)/1.0155e-05

-0.011225997045790177

In [8]:
alpha =  0.050354/0.0455
beta =  0.069360/0.0631
abs(alpha/beta-1)

0.006799181210945804

In [10]:
# 0.89
BDT_val = 0.89
eff_sig_cal =  0.050354
eff_ref_cal =  0.069360

Nsig_err, Nsig, Nref_err, Nref = 48.1649872857858, 583.8742593051962, 225.2143202050156, 24508.46923632544 

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_0_var, br_0_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_0_var)/orig_val

print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference: {(orig_val - br_0_var):.6f}")

err_of_difference = math.sqrt(abs(br_0_err*br_0_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_0_var)/err_of_difference*100:.6f}%")


BDT = 0.89
signal eff error: 8.927343e-05, ref eff error: 1.037217e-04
Mode: eta -> gg Central value = 1.2371e-04
Mode: eta -> gg Statistical uncertainty = 1.0269e-05
Mode: eta -> gg Pull = -0.1252
Mode: eta -> gg stas. unc./Central value = 8.3002e-02

Br_ratio value = 3.2815e-02
Br_ratio Statistical uncertainty = 2.7238e-03
Difference from orig.: 0.6714%
Difference: 0.000222
err_of_Difference: 0.00040568
Difference/err_of_Difference: 54.678465%


In [11]:
# 0.90
BDT_val = 0.90
eff_sig_cal =  0.048080
eff_ref_cal =  0.066429

Nsig_err, Nsig, Nref_err, Nref = 45.79657150375624, 558.3119907831142, 218.0061577806482, 23526.46810629756

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_1_var, br_1_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_1_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference: {(orig_val - br_1_var):.6f}")

err_of_difference = math.sqrt(abs(br_1_err*br_1_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_1_var)/err_of_difference*100:.6f}%")


BDT = 0.9
signal eff error: 8.733872e-05, ref eff error: 1.016663e-04
Mode: eta -> gg Central value = 1.2361e-04
Mode: eta -> gg Statistical uncertainty = 1.0204e-05
Mode: eta -> gg Pull = -0.1362
Mode: eta -> gg stas. unc./Central value = 8.2549e-02

Br_ratio value = 3.2788e-02
Br_ratio Statistical uncertainty = 2.7066e-03
Difference from orig.: 0.7548%
Difference: 0.000249
err_of_Difference: 0.00026716
Difference/err_of_Difference: 93.340756%


In [12]:
# 0.92
BDT_val = 0.92
eff_sig_cal =  0.042650
eff_ref_cal =  0.059375

Nsig_err, Nsig, Nref_err, Nref = 40.20117141803934, 492.4599855124414, 201.63626848276908, 21085.593843786162

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_2_var, br_2_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_2_var)/br_2_var
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference: {(orig_val - br_2_var):.6f}")

err_of_difference = math.sqrt(abs(br_2_err*br_2_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_2_var)/err_of_difference*100:.6f}%")


BDT = 0.92
signal eff error: 8.249341e-05, ref eff error: 9.647937e-05
Mode: eta -> gg Central value = 1.2258e-04
Mode: eta -> gg Statistical uncertainty = 1.0075e-05
Mode: eta -> gg Pull = -0.2404
Mode: eta -> gg stas. unc./Central value = 8.2192e-02

Br_ratio value = 3.2514e-02
Br_ratio Statistical uncertainty = 2.6724e-03
Difference from orig.: 1.6095%
Difference: 0.000523
err_of_Difference: 0.00033571
Difference/err_of_Difference: 155.881129%


In [13]:
# 0.93
BDT_val = 0.93
eff_sig_cal =  0.039363
eff_ref_cal =  0.055124

Nsig_err, Nsig, Nref_err, Nref = 37.09774949527011, 453.324856470759, 192.20798131017, 19654.166912562774

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_3_var, br_3_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_3_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference: {(orig_val - br_3_var):.6f}")

err_of_difference = math.sqrt(abs(br_3_err*br_3_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_3_var)/err_of_difference*100:.6f}%")


BDT = 0.93
signal eff error: 7.938677e-05, ref eff error: 9.317130e-05
Mode: eta -> gg Central value = 1.2177e-04
Mode: eta -> gg Statistical uncertainty = 1.0036e-05
Mode: eta -> gg Pull = -0.3216
Mode: eta -> gg stas. unc./Central value = 8.2417e-02

Br_ratio value = 3.2300e-02
Br_ratio Statistical uncertainty = 2.6621e-03
Difference from orig.: 2.2305%
Difference: 0.000737
err_of_Difference: 0.00040926
Difference/err_of_Difference: 180.055725%


### eta -> pipipi

In [14]:
mode="pipipi"
# 0.92
eff_sig_cal =  0.0381
eff_ref_cal = 0.052408

Nsig_err, Nsig, Nref_err, Nref = 23.288942474989838, 264.94003796983776 , 123.61477929656485 , 11273.896780101566

print(f"Original values")
orig_val, orig_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)
print(f"Original values: {orig_val} pm {orig_err}")

Original values
signal eff error: 7.815411e-05, ref eff error: 9.097747e-05
Mode: eta -> pipipi Central value = 1.2187e-04
Mode: eta -> pipipi Statistical uncertainty = 1.0795e-05
Mode: eta -> pipipi Pull = -0.2902
Mode: eta -> pipipi stas. unc./Central value = 8.8584e-02

Br_ratio value = 3.2326e-02
Br_ratio Statistical uncertainty = 2.8635e-03
Original values: 0.032325566958405876 pm 0.0028635248432211085


In [15]:
# 0.90
BDT_val = 0.90
eff_sig_cal =  0.042260
eff_ref_cal =  0.057713

Nsig_err, Nsig, Nref_err, Nref = 25.684454000550602, 288.4344612049827, 131.48464018921095, 12434.08110576568

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_0_var, br_0_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_0_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_0_var)/br_0_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_0_err*br_0_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_0_var)/err_of_difference*100:.6f}%")

BDT = 0.9
signal eff error: 8.213210e-05, ref eff error: 9.520347e-05
Mode: eta -> pipipi Central value = 1.1943e-04
Mode: eta -> pipipi Statistical uncertainty = 1.0710e-05
Mode: eta -> pipipi Pull = -0.5199
Mode: eta -> pipipi stas. unc./Central value = 8.9673e-02

Br_ratio value = 3.1679e-02
Br_ratio Statistical uncertainty = 2.8408e-03
Difference from orig.: 1.9988%
Difference/current_br_err: 22.7442%
err_of_Difference: 0.00035999
Difference/err_of_Difference: 179.480634%


In [16]:
# 0.91
BDT_val = 0.91
eff_sig_cal =  0.040328
eff_ref_cal =  0.055237

Nsig_err, Nsig, Nref_err, Nref = 24.43622142138041, 273.47624546581096, 127.77157359817465, 11888.06782921226

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_1_var, br_1_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_1_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_1_var)/br_1_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_1_err*br_1_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_1_var)/err_of_difference*100:.6f}%")

BDT = 0.91
signal eff error: 8.031361e-05, ref eff error: 9.326117e-05
Mode: eta -> pipipi Central value = 1.1879e-04
Mode: eta -> pipipi Statistical uncertainty = 1.0691e-05
Mode: eta -> pipipi Pull = -0.5811
Mode: eta -> pipipi stas. unc./Central value = 8.9998e-02

Br_ratio value = 3.1509e-02
Br_ratio Statistical uncertainty = 2.8357e-03
Difference from orig.: 2.5267%
Difference/current_br_err: 28.8030%
err_of_Difference: 0.00039798
Difference/err_of_Difference: 205.229233%


In [17]:
# 0.93
BDT_val = 0.93
eff_sig_cal =  0.035525
eff_ref_cal =  0.049122

Nsig_err, Nsig, Nref_err, Nref = 21.831404137952276, 245.01929444317085, 118.91272654301974, 10575.209182542412

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_2_var, br_2_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_2_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_2_var)/br_2_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_2_err*br_2_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_2_var)/err_of_difference*100:.6f}%")


BDT = 0.93
signal eff error: 7.556782e-05, ref eff error: 8.823173e-05
Mode: eta -> pipipi Central value = 1.2078e-04
Mode: eta -> pipipi Statistical uncertainty = 1.0847e-05
Mode: eta -> pipipi Pull = -0.3891
Mode: eta -> pipipi stas. unc./Central value = 8.9807e-02

Br_ratio value = 3.2037e-02
Br_ratio Statistical uncertainty = 2.8772e-03
Difference from orig.: 0.8924%
Difference/current_br_err: 10.0259%
err_of_Difference: 0.00027990
Difference/err_of_Difference: 103.060072%


In [18]:
# 0.94
BDT_val = 0.94
eff_sig_cal =  0.032553
eff_ref_cal =  0.045345

Nsig_err, Nsig, Nref_err, Nref = 20.40927245692299, 226.68384808420643, 113.50441949422748, 9767.11383475488

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_3_var, br_3_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_3_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_3_var)/br_3_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_3_err*br_3_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_3_var)/err_of_difference*100:.6f}%")


BDT = 0.94
signal eff error: 7.244918e-05, ref eff error: 8.494001e-05
Mode: eta -> pipipi Central value = 1.2188e-04
Mode: eta -> pipipi Statistical uncertainty = 1.1064e-05
Mode: eta -> pipipi Pull = -0.2819
Mode: eta -> pipipi stas. unc./Central value = 9.0781e-02

Br_ratio value = 3.2329e-02
Br_ratio Statistical uncertainty = 2.9349e-03
Difference from orig.: -0.0107%
Difference/current_br_err: -0.1181%
err_of_Difference: 0.00064315
Difference/err_of_Difference: -0.538926%


## Br(Ds+ -> eta K+)

### eta -> gg

In [19]:
# Constants
Br_ref_dec = 0.017000000
Br_sig_dec = 0.001600000

# Br_sig_PDG = 0.001730000
# Br_sig_PDG_err = 0.000080000

mode="gg"
# 0.91
eff_sig_cal = 0.0386
eff_ref_cal = 0.054430

Nsig_err, Nsig, Nref_err, Nref = 73.30410080842262, 3127.0525804656004, 242.4697212229985 , 47273.768092928294

print(f"Original values")
orig_val, orig_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

Original values
signal eff error: 7.864481e-05, ref eff error: 9.261693e-05
Mode: eta -> gg Central value = 1.5857e-03
Mode: eta -> gg Statistical uncertainty = 3.8051e-05
Mode: eta -> gg Pull = -0.3764
Mode: eta -> gg stas. unc./Central value = 2.3996e-02

Br_ratio value = 9.3275e-02
Br_ratio Statistical uncertainty = 2.2383e-03


In [21]:
# 0.89
BDT_val = 0.89
eff_sig_cal = 0.043635
eff_ref_cal = 0.060957

Nsig_err, Nsig, Nref_err, Nref = 79.67260731175907, 3486.888835128606, 259.12026889503613, 52895.99975277429

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_0_var, br_0_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_0_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_0_var)/br_0_err:.4f}sigma")
err_of_difference = math.sqrt(abs(br_0_err*br_0_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_0_var)/err_of_difference*100:.6f}%")

BDT = 0.89
signal eff error: 8.339763e-05, ref eff error: 9.767399e-05
Mode: eta -> gg Central value = 1.5655e-03
Mode: eta -> gg Statistical uncertainty = 3.6583e-05
Mode: eta -> gg Pull = -0.9431
Mode: eta -> gg stas. unc./Central value = 2.3368e-02

Br_ratio value = 9.2088e-02
Br_ratio Statistical uncertainty = 2.1520e-03
Difference from orig.: 1.2726%
Difference/current_br_err: 0.5516sigma
err_of_Difference: 0.00061560
Difference/err_of_Difference: 192.816704%


In [22]:
# 0.90
BDT_val = 0.90
eff_sig_cal =  0.041261
eff_ref_cal =  0.057861

Nsig_err, Nsig, Nref_err, Nref = 76.61677748781335, 3317.111265409061, 251.2169099814564, 50226.003292920825

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_1_var, br_1_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_1_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_1_var)/br_1_err*100:.4f}%")
err_of_difference = math.sqrt(abs(br_1_err*br_1_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_1_var)/err_of_difference*100:.6f}%")

BDT = 0.9
signal eff error: 8.119783e-05, ref eff error: 9.531798e-05
Mode: eta -> gg Central value = 1.5744e-03
Mode: eta -> gg Statistical uncertainty = 3.7208e-05
Mode: eta -> gg Pull = -0.6869
Mode: eta -> gg stas. unc./Central value = 2.3633e-02

Br_ratio value = 9.2614e-02
Br_ratio Statistical uncertainty = 2.1887e-03
Difference from orig.: 0.7086%
Difference/current_br_err: 30.1980%
err_of_Difference: 0.00046832
Difference/err_of_Difference: 141.133028%


In [23]:
# 0.92
BDT_val = 0.92
eff_sig_cal =  0.035748
eff_ref_cal =  0.050634

Nsig_err, Nsig, Nref_err, Nref = 69.47289189914841, 2913.7264662368107, 232.70446129182892, 43989.51955163527

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_2_var, br_2_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_2_var)/br_2_var
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_2_var)/br_2_err*100:.4f}%")
err_of_difference = math.sqrt(abs(br_2_err*br_2_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_2_var)/err_of_difference*100:.6f}%")

BDT = 0.92
signal eff error: 7.579587e-05, ref eff error: 8.950810e-05
Mode: eta -> gg Central value = 1.5949e-03
Mode: eta -> gg Statistical uncertainty = 3.8953e-05
Mode: eta -> gg Pull = -0.1304
Mode: eta -> gg stas. unc./Central value = 2.4423e-02

Br_ratio value = 9.3819e-02
Br_ratio Statistical uncertainty = 2.2913e-03
Difference from orig.: -0.5795%
Difference/current_br_err: -23.7275%
err_of_Difference: 0.00049030
Difference/err_of_Difference: -110.886706%


In [24]:
# 0.93
BDT_val = 0.93
eff_sig_cal =  0.032568
eff_ref_cal =  0.046443

Nsig_err, Nsig, Nref_err, Nref = 65.18894250079484, 2648.0989172660225, 221.75916441719164, 40395.098270714414

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_3_var, br_3_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_3_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_3_var)/br_3_err*100:.4f}%")
err_of_difference = math.sqrt(abs(br_3_err*br_3_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_3_var)/err_of_difference*100:.6f}%")

BDT = 0.93
signal eff error: 7.246531e-05, ref eff error: 8.591279e-05
Mode: eta -> gg Central value = 1.5892e-03
Mode: eta -> gg Statistical uncertainty = 4.0083e-05
Mode: eta -> gg Pull = -0.2690
Mode: eta -> gg stas. unc./Central value = 2.5222e-02

Br_ratio value = 9.3483e-02
Br_ratio Statistical uncertainty = 2.3578e-03
Difference from orig.: -0.2233%
Difference/current_br_err: -8.8338%
err_of_Difference: 0.00074129
Difference/err_of_Difference: -28.097916%


### eta -> pipipi

In [25]:
mode="pipipi"
# 0.92
eff_sig_cal  =  0.027621
eff_ref_cal =  0.039159

Nsig_err, Nsig, Nref_err, Nref = 41.070609843501074 , 1310.6394390681576 , 149.0261848133723,19798.445187596266


print(f"Original values")
orig_val, orig_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

Original values
signal eff error: 6.690551e-05, ref eff error: 7.918920e-05
Mode: eta -> pipipi Central value = 1.5955e-03
Mode: eta -> pipipi Statistical uncertainty = 5.1419e-05
Mode: eta -> pipipi Pull = -0.0878
Mode: eta -> pipipi stas. unc./Central value = 3.2228e-02

Br_ratio value = 9.3852e-02
Br_ratio Statistical uncertainty = 3.0246e-03


In [26]:
# 0.90
BDT_val = 0.90
eff_sig_cal =  0.032139
eff_ref_cal = 0.045093

Nsig_err, Nsig, Nref_err, Nref = 44.829483405143606, 1508.4335744618425, 160.32647849046225, 22749.601902138795

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_0_var, br_0_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_0_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_0_var)/br_0_err*100:.4f}%")
err_of_difference = math.sqrt(abs(br_0_err*br_0_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_0_var)/err_of_difference*100:.6f}%")

BDT = 0.9
signal eff error: 7.200241e-05, ref eff error: 8.471484e-05
Mode: eta -> pipipi Central value = 1.5815e-03
Mode: eta -> pipipi Statistical uncertainty = 4.8305e-05
Mode: eta -> pipipi Pull = -0.3823
Mode: eta -> pipipi stas. unc./Central value = 3.0543e-02

Br_ratio value = 9.3031e-02
Br_ratio Statistical uncertainty = 2.8415e-03
Difference from orig.: 0.8746%
Difference/current_br_err: 28.8883%
err_of_Difference: 0.00103651
Difference/err_of_Difference: 79.194679%


In [27]:
# 0.91
BDT_val = 0.91
eff_sig_cal =  0.029983
eff_ref_cal = 0.042320

Nsig_err, Nsig, Nref_err, Nref = 43.08963333545432, 1414.2747661899946, 155.04352139442017, 21352.836571546723

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_1_var, br_1_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_1_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_1_var)/br_1_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_1_err*br_1_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_1_var)/err_of_difference*100:.6f}%")

BDT = 0.91
signal eff error: 6.962282e-05, ref eff error: 8.218781e-05
Mode: eta -> pipipi Central value = 1.5893e-03
Mode: eta -> pipipi Statistical uncertainty = 4.9777e-05
Mode: eta -> pipipi Pull = -0.2156
Mode: eta -> pipipi stas. unc./Central value = 3.1321e-02

Br_ratio value = 9.3486e-02
Br_ratio Statistical uncertainty = 2.9281e-03
Difference from orig.: 0.3897%
Difference/current_br_err: 12.4894%
err_of_Difference: 0.00075813
Difference/err_of_Difference: 48.236998%


In [28]:
# 0.93
BDT_val = 0.93
eff_sig_cal =  0.024982
eff_ref_cal = 0.035699

Nsig_err, Nsig, Nref_err, Nref = 38.750694192239166, 1182.526465994415, 142.04365758234053, 18060.10829658532

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_2_var, br_2_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_2_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_2_var)/br_2_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_2_err*br_2_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_2_var)/err_of_difference*100:.6f}%")

BDT = 0.93
signal eff error: 6.371538e-05, ref eff error: 7.574583e-05
Mode: eta -> pipipi Central value = 1.5906e-03
Mode: eta -> pipipi Statistical uncertainty = 5.3604e-05
Mode: eta -> pipipi Pull = -0.1749
Mode: eta -> pipipi stas. unc./Central value = 3.3700e-02

Br_ratio value = 9.3566e-02
Br_ratio Statistical uncertainty = 3.1532e-03
Difference from orig.: 0.3046%
Difference/current_br_err: 9.0668%
err_of_Difference: 0.00089116
Difference/err_of_Difference: 32.081159%


In [29]:
# 0.94
BDT_val = 0.94
eff_sig_cal =  0.022074
eff_ref_cal = 0.031845

Nsig_err, Nsig, Nref_err, Nref = 36.095607222288436, 1043.9757425430848, 133.8259622675605, 16096.82596891294

print(f"BDT = {BDT_val}")
temp_var, temp_err = cal_ratio_Br(mode, Br_ref_dec, Br_sig_dec, eff_sig_cal, eff_ref_cal, Nsig_err, Nsig, Nref_err, Nref)

br_3_var, br_3_err = temp_var, temp_err 
diff_from_orig = (orig_val - br_3_var)/orig_val
print(f"Difference from orig.: {diff_from_orig*100:.4f}%")
print(f"Difference/current_br_err: {(orig_val - br_3_var)/br_3_err*100:.4f}%")

err_of_difference = math.sqrt(abs(br_3_err*br_3_err - orig_err*orig_err ))
print(f"err_of_Difference: {err_of_difference:.8f}")
print(f"Difference/err_of_Difference: {(orig_val - br_3_var)/err_of_difference*100:.6f}%")

BDT = 0.94
signal eff error: 5.998158e-05, ref eff error: 7.168321e-05
Mode: eta -> pipipi Central value = 1.5906e-03
Mode: eta -> pipipi Statistical uncertainty = 5.6563e-05
Mode: eta -> pipipi Pull = -0.1663
Mode: eta -> pipipi stas. unc./Central value = 3.5561e-02

Br_ratio value = 9.3564e-02
Br_ratio Statistical uncertainty = 3.3272e-03
Difference from orig.: 0.3067%
Difference/current_br_err: 8.6508%
err_of_Difference: 0.00138632
Difference/err_of_Difference: 20.762252%
